In [319]:
import os
import tarfile
from pathlib import Path

import polars as pl
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split
from polars import selectors as cs


In [320]:
raw_path = Path(os.environ["DATA_DIR"], "raw", "mbd_mini")
raw_path.mkdir(exist_ok=True)
snapshot_download(
    repo_id="ai-lab/MBD-mini",
    repo_type="dataset",
    local_dir=raw_path,
)


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

'/mnt/data/raw/mbd_mini'

In [321]:
if not (raw_path / "detail").exists():
    with tarfile.open(raw_path / "detail.tar.gz", "r:gz") as tar:
        tar.extractall(raw_path)

In [ ]:
if not (raw_path / "targets").exists():
    with tarfile.open(raw_path / "targets.tar.gz", "r:gz") as tar:
        tar.extractall(raw_path)


In [323]:
trx = pl.read_parquet(raw_path / "detail" / "trx" / "fold=0").drop_nulls()
target = pl.read_parquet(raw_path / "targets" / "fold=0").drop_nulls()

In [324]:
target_mon = target.with_columns(
    mon=pl.col("mon").str.split("-").list.slice(0, 2).list.join("-")
)
trx_mon = trx.with_columns(mon=pl.col("event_time").dt.strftime("%Y-%m"))


In [328]:
trx_filt = trx_mon.filter(
    (pl.col("event_time").max() - pl.col("event_time").min()).over("client_id", "mon")
    > pl.duration(weeks=1),
    pl.len().over("client_id", "mon").is_between(32, 1024),
    pl.col("currency") == 11,
).drop("src_type31", "src_type21", "currency")

In [329]:
trx_proc = trx_filt.with_columns(
    cs.integer().rank("dense").cast(pl.Int32),
    amount=pl.col("amount").abs().log1p() * pl.col("amount").sign(),
    time=(pl.col("event_time") - pl.col("event_time").min()).over("client_id", "mon")
    / (pl.col("event_time").max() - pl.col("event_time").min())
    .over("client_id", "mon")
    .median(),
).drop("event_time")

In [330]:
target_proc = target_mon.with_columns(pl.col("^target_.$").cast(pl.Boolean))

In [331]:
trx_gb = trx_proc.sort("client_id", "time").group_by("client_id", "mon").agg(pl.all())

In [332]:
df = trx_gb.join(target_proc, on=["client_id", "mon"]).drop("client_id", "mon")

In [333]:
# Ignore 2nd target and multi-target due to extreme imbalance
df_targ = df.with_columns(
    target=pl.when("target_1")
    .then(1)
    .when("target_3")
    .then(2)
    .when("target_4")
    .then(3)
    .otherwise(0)
).drop("^target_.$")

In [334]:
train_val, test = train_test_split(df_targ, test_size=0.2)
train, val = train_test_split(train_val, test_size=0.2)

train = train.with_columns(split=pl.lit("train"))
val = val.with_columns(split=pl.lit("val"))
test = test.with_columns(split=pl.lit("test"))
df_final = pl.concat([train, val, test])

In [338]:
df_final.select(pl.col(pl.List(pl.Int32)).list.explode().n_unique()).row(0, named=True)

{'event_type': 52,
 'event_subtype': 56,
 'src_type11': 36,
 'src_type12': 125,
 'dst_type11': 42,
 'dst_type12': 157,
 'src_type22': 82,
 'src_type32': 81}

In [ ]:
df_final.write_parquet(
    Path(os.environ["DATA_DIR"], "preprocessed", "mbd_micro.parquet")
)